# Lesson 6 — k-NN, Naive Bayes and Support Vector Machines

Self-assessment. No code: every answer is a sentence, a short calculation, or a
diagnosis.

As in lesson 5, many of these questions describe a situation and ask you to
*criticise* it. Those are the ones worth your time. Knowing which family to
reach for — and being able to say why the other two would fail — is what this
lesson is for, and it is what the final project is marked on.

Numbers quoted throughout come from the lesson's notebooks: 1,200 industrial
pumps, 61.3% of them faulty, with 4% of the labels deliberately flipped. Scores
are cross-validated accuracy, and every one of them should be read against the
majority baseline of **0.613** and the noise ceiling of about **0.96**. Support
vector machines (SVM), and the radial basis function (RBF) kernel they are
usually run with, arrive in Part 6.

## Part 1 — A problem no straight line solves

**1. Logistic regression cross-validated at 0.613 ± 0.000 on the 1,200 pumps, and the majority baseline is 0.613 exactly. A colleague reads this as "the linear model matches the baseline, so this is a hard problem". Correct them: what has the model actually done, and which part of the printed result gives it away?**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>It has not been narrowly beaten by the baseline — it <b>is</b> the baseline. It predicts <b>faulty for every pump</b>, because with a straight boundary that is genuinely its best available answer.</li>
        <li>The giveaway is <b>± 0.000</b>. A model that gives every row the same answer is perfectly consistent across folds, so the spread collapses to zero. A genuine 0.613 would wobble from fold to fold.</li>
        <li>The problem is not hard: k-NN reaches <b>0.944</b> and an RBF SVM <b>0.947</b> on the same 1,200 rows. It is the wrong <b>shape</b> for a straight boundary, which is an entirely different complaint and has an entirely different remedy.</li>
    </ul>
    </p>
</details>

**2. Four per cent of the pump labels are deliberately flipped, so no model can exceed roughly 0.96. Name what that number is, and state what you should conclude about a model that cross-validates at 0.985 on this data.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>It is the <b>noise ceiling</b> — lesson 5's noise floor arriving in a classification problem. It is a property of the data, not of any model, and no algorithm and no quantity of extra data gets past it.</li>
        <li>A score of 0.985 is therefore <b>evidence of contamination, not of excellence</b>: leakage, a row duplicated across folds, or a target that has crept into the features.</li>
        <li>It is also the yardstick for every score below it. k-NN's 0.944 and the RBF SVM's 0.947 sit close to the ceiling, so most of what remains is flipped labels rather than room for improvement.</li>
    </ul>
    </p>
</details>

**3. A pump is faulty when its readings fall outside its design envelope — too low as well as too high. Explain how that one physical fact, stated before anything is fitted, predicts the whole table of results at the end of the lesson.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>The healthy pumps form a <b>disc</b> around the design point and the faulty ones the <b>annulus around them</b>. One class encloses the other.</li>
        <li>No straight line separates an enclosed class from the class enclosing it, so every strictly linear method scores the base rate: logistic regression 0.613, linear SVM 0.613. Choosing the <i>best</i> straight line does not help when no straight line works.</li>
        <li>Anything that can draw a local or a curved boundary succeeds: Naive Bayes 0.933, k-NN 0.944, RBF SVM 0.947.</li>
        <li>This is lesson 2's argument for looking at the data first, with a price attached. <b>The scatter plot tells you which family to reach for</b> before a single model is fitted.</li>
    </ul>
    </p>
</details>

## Part 2 — k-nearest neighbours

**4. State the whole of the k-nearest neighbours algorithm in one sentence, and explain why it is called a lazy learner.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>To classify a new point, find the <code>k</code> training points closest to it and take the <b>majority vote</b>. That is all of it.</li>
        <li>There is <b>no training step</b>: "fitting" means storing the data. Hence lazy — an unusually honest name for an algorithm.</li>
        <li>Nothing is free, though. The work simply moves to prediction time, at <code>O(mn)</code> per query for <code>m</code> stored rows and <code>n</code> features.</li>
    </ul>
    </p>
</details>

**5. Vibration runs over tens of Hz and pressure over a couple of bar. Explain what happens to k-NN if those two features are not put on a common scale.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>Euclidean distance <code>d(x, x') = sqrt(sum_j (x_j - x'_j)^2)</code> adds each feature's squared difference <b>in that feature's own units</b>.</li>
        <li>A feature ranging over tens contributes squared differences hundreds of times larger than one ranging over units, so <b>vibration alone decides every neighbour</b> and pressure is effectively discarded. You have a one-dimensional model and no warning that you have one.</li>
        <li>Lesson 2 argued for scaling as hygiene. Here it is not hygiene: <b>distance is the entire model</b>, so the choice of units changes what the model is, not merely how quickly it trains.</li>
    </ul>
    </p>
</details>

**6. At k = 1 the training accuracy on the pumps is exactly 1.000 while the cross-validated score is 0.912; at k = 5 the cross-validated score peaks at 0.944. State what the training score at k = 1 measures, and what k = 5 is balancing.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li><b>Nothing.</b> At k = 1 the training accuracy is exactly 1.000 on any dataset whatever, because every training point is its own nearest neighbour. It is the purest illustration of lesson 5's point that a training score is not a measurement.</li>
        <li>k is the <b>bias-variance dial</b> made visible, and you turn it with one integer. Small k: the boundary follows every point, including the 4% that are mislabelled — low bias, high variance.</li>
        <li>Large k: the vote is taken over a wide neighbourhood, so the boundary smooths and eventually stops following real structure — high bias, low variance.</li>
        <li>k = 5 sits where the neighbourhood is <b>wide enough to average out the flipped labels and narrow enough to still follow the boundary</b>.</li>
    </ul>
    </p>
</details>

**7. At k = 401 the pumps score 0.828 training and 0.708 cross-validated. A student concludes that the model has "collapsed to predicting the majority class". Why is that wrong, and what has actually happened to the boundary?**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>If it had collapsed to one class it would score <b>0.613</b>, the majority baseline, in both columns. 0.708 is above that, so the data is still doing some work.</li>
        <li>What has happened is that the boundary has <b>inflated past the true envelope</b>: with 401 votes drawn from a wide neighbourhood, the disc of healthy pumps swells and swallows faulty pumps near its edge.</li>
        <li>It has not stopped predicting; it has stopped <b>following</b> the boundary and started averaging over it. That is high bias, and it is visible in the k = 401 panel of the boundary figure as a disc distinctly larger than the one that generated the data.</li>
    </ul>
    </p>
</details>

**8. k-NN needs no training time at all. Name the two costs you pay instead — one computational, one not.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li><b>Prediction costs O(mn) per query</b>, because a naive implementation compares the query against every stored row. For 1,200 pumps that is nothing; for ten million rows answering a thousand queries a second it is the entire engineering problem, and it is why approximate nearest-neighbour indexes are an industry.</li>
        <li><b>The model is the dataset.</b> You cannot ship the model without shipping the training data, which is a legal and privacy question as much as a practical one — and a reason k-NN is a poor fit wherever the data cannot leave the building.</li>
    </ul>
    </p>
</details>

## Part 3 — The curse of dimensionality

**9. Adding 50 columns of pure noise took k-NN from 0.938 to 0.602 while the two real readings were left untouched. Given that the signal never left, explain precisely why the score fell.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>Every added column contributes its own squared difference to <b>every</b> distance. Fifty noise terms swamp the two informative ones, so the ranking of neighbours is decided mostly by noise.</li>
        <li>"Nearest" then stops selecting points that are near <i>in the ways that matter</i>, and the vote is taken among near-arbitrary rows.</li>
        <li>The mechanism is <b>geometry, not statistics</b>. The problem is exactly as solvable as it was: a method that used only the first two columns would be entirely unaffected.</li>
        <li>At 52 columns the score is 0.602, <b>below the 0.613 baseline</b> — a model that ignored the data completely would now do better.</li>
    </ul>
    </p>
</details>

**10. The mean ratio of nearest to farthest distance is 0.016 in 2 dimensions and 0.701 in 100. What does each number say about what the word "nearest" is worth?**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>In two dimensions the nearest point is about <b>2% as far away as the farthest</b>. "Nearest" is a strong claim, and that gap is exactly what makes the word mean anything.</li>
        <li>In one hundred dimensions the nearest point is <b>70% as far away as the farthest</b>. It is barely nearer than a random one, so a vote among "the five nearest" is close to a vote among five taken at random.</li>
        <li>The ratio climbs monotonically towards 1 — 0.263 at 10 dimensions, 0.592 at 50, 0.855 at 500 — and at 1 every point is the same distance from every other and distance carries no information at all.</li>
    </ul>
    </p>
</details>

**11. State what the curse of dimensionality is not, and name two methods besides k-NN that it reaches.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>It is <b>not</b> that high-dimensional problems are inherently unlearnable. Lesson 9's neural networks work happily in thousands of dimensions.</li>
        <li>It is that <b>methods built on distance lose their footing</b>, because the quantity they depend on stops varying between candidates.</li>
        <li>So it reaches <b>k-means</b> (lesson 8) and <b>RBF kernels</b>, which are a function of <code>||x - x'||^2</code> and degrade for the same reason. Anything whose answer is a function of pairwise distance inherits the problem; anything that learns which directions matter does not.</li>
    </ul>
    </p>
</details>

**12. A colleague adds thirty plausible-looking sensor channels to a k-NN model, reasoning that "extra features can only help — the useless ones will just be ignored". Explain why the instinct is sound and where exactly it fails.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>The instinct is sound <b>for a linear model</b>, where an irrelevant feature costs you a coefficient near zero and very little else. That is good practice, and it is how the habit forms.</li>
        <li>With k-NN an irrelevant feature costs you <b>a dimension in the distance</b>, and dimensions are what the method is made of. There is no coefficient to send to zero.</li>
        <li>The measured price: 25 noise columns take the pumps from 0.938 to 0.662, and 50 take them to 0.602, under the baseline. <b>The same habit, transferred one lesson later, does real damage.</b></li>
        <li>The remedy is selection or projection <b>before</b> the distance is computed — not more neighbours, which cannot recover a distance that has already been diluted.</li>
    </ul>
    </p>
</details>

**13. With 100 noise columns the pumps score 0.578. A colleague says "it is still 58% accurate, so it is still learning something". Explain what is wrong with that reading, and state the right reference point.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>0.578 is <b>below the majority baseline of 0.613</b>. A rule that ignored every column and always answered "faulty" would score higher, so the features are not merely useless, they are actively misleading the vote.</li>
        <li>Accuracy above zero is not evidence of skill. The reference point is the <b>no-skill rate on this class balance</b>, which is 0.613 here and not 0.500.</li>
        <li>Reading accuracy against 0.5 rather than against the baseline is what makes an imbalanced dataset flatter every model fitted to it — the more imbalanced, the more flattering.</li>
    </ul>
    </p>
</details>

## Part 4 — Bayes' rule and the naive assumption

**14. Write Bayes' rule for classification, and identify which of its quantities is the difficult one and why.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li><code>P(y = c | x) = P(x | y = c) * P(y = c) / P(x)</code>.</li>
        <li><code>P(y = c)</code> is the prior, and it is a count. <code>P(x)</code> is <b>identical across classes</b>, so it cannot change which class wins and can be dropped from the comparison entirely.</li>
        <li>Everything hard is in <code>P(x | y = c)</code>: the probability of <b>this exact combination of readings</b> among examples of that class. With two features that is a two-dimensional density and you can estimate it; with twenty it is a twenty-dimensional one, and no quantity of data populates a twenty-dimensional space.</li>
        <li>That is the curse of Part 3 arriving from a completely different direction — density estimation this time rather than distance.</li>
    </ul>
    </p>
</details>

**15. State the naive assumption precisely, and say what it buys.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li><b>Given the class, the features are independent of one another.</b> The words "given the class" carry the whole statement and are the half most often dropped.</li>
        <li>If it holds, the joint density factorises: <code>P(x | y = c) = prod_j P(x_j | y = c)</code>, and the classifier becomes <code>argmax_c P(y = c) * prod_j P(x_j | y = c)</code>.</li>
        <li>What it buys: <b>one n-dimensional estimation problem becomes n one-dimensional ones</b>. Training is a single pass computing a mean and a variance per feature per class, it needs very little data per feature, and adding features costs almost nothing.</li>
    </ul>
    </p>
</details>

**16. In practice the product over features is computed as a sum of logarithms. Explain why, and why the answer is unchanged.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>A product of hundreds of small probabilities <b>underflows to exactly zero in floating point</b>, and every class then ties at zero. It is the same problem lesson 4 met with the log-likelihood, and the same fix.</li>
        <li>Taking logarithms turns the product into <code>log P(y = c) + sum_j log P(x_j | y = c)</code>, a sum of moderate negative numbers that no arithmetic ruins.</li>
        <li>The answer is unchanged because the logarithm is <b>monotonically increasing</b>: it preserves the ordering of the classes, and therefore the argmax, even though it destroys the values themselves.</li>
    </ul>
    </p>
</details>

**17. On the pumps the correlation between the two readings is −0.046 overall, −0.006 within healthy pumps and −0.049 within faulty ones. Which of those three numbers tests the naive assumption, and what do they tell you that the accuracy of 0.933 does not?**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>The <b>within-class</b> ones. The assumption concerns independence <i>given the class</i>, which is not the same as independence overall — two well-separated classes can make strongly dependent features look uncorrelated in aggregate, and the reverse happens too.</li>
        <li>At <b>−0.006</b> and −0.049 the readings are essentially uncorrelated within each class, so on this dataset the assumption is very nearly true and Naive Bayes is not getting away with anything.</li>
        <li>The accuracy tells you the model worked <i>here</i>. The correlations tell you <b>why</b>, and therefore whether to expect it to keep working on the next fleet. That is the half that transfers.</li>
    </ul>
    </p>
</details>

## Part 5 — Where Naive Bayes fails, and what its probabilities are worth

**18. A second pair of sensors flags a pump when exactly one of the two readings is high. Naive Bayes scores 0.404 against a 0.523 baseline; k-NN scores 0.967 and an RBF SVM 0.972. Explain why the same data is easy for one method and impossible for the other.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>The signal lives <b>entirely in the interaction</b>: both readings high is the designed high-load mode, both low is idle, and one without the other is a mismatch between demand and delivery. Seen jointly there are four clear groups and a perfectly learnable rule.</li>
        <li>Naive Bayes never sees the joint distribution. It sees <b>one feature at a time</b>, and on each sensor alone the two classes sit almost exactly on top of one another — the marginals are uninformative by construction.</li>
        <li>k-NN and a kernel both work in the joint space, where the four groups are plainly separated, so they reach the ceiling.</li>
        <li><b>No quantity of data repairs this.</b> The failure is in the hypothesis class, not in the sample: the model cannot represent what is being asked of it. Both linear models fail here too — logistic regression 0.393, linear SVM 0.606.</li>
    </ul>
    </p>
</details>

**19. 0.404 is below chance. A model with no usable signal ought to sit at 0.5. Explain why it does not.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>Because it is not guessing. The class means on each sensor differ by about <b>0.17 against a spread near 1</b>, purely as an artefact of a finite sample.</li>
        <li>That accident is the <b>only</b> per-feature evidence available, Naive Bayes has nothing else to multiply, and in this sample it points the wrong way — consistently, on row after row, which is what turns a small bias into a systematic error.</li>
        <li><b>A model with no signal does not sit politely at 50%. It follows whatever spurious structure it can find.</b></li>
        <li>And 0.5 was never the right reference in any case: the baseline on this data is 0.523, and 0.404 is well below that too.</li>
    </ul>
    </p>
</details>

**20. Mean confidence when correct 0.567, mean confidence when wrong 0.555. Name the property being measured, and state what follows for how the output may be used.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li><b>Calibration.</b> A calibrated model's reported probability matches its observed frequency of being right. Here the two means are a hair apart, so <b>the number carries almost no information about whether the answer is right</b>.</li>
        <li>Lesson 5's distinction applies exactly: <b>the ranking may be useful while the probabilities are not.</b></li>
        <li>What follows: never feed a Naive Bayes posterior into anything that treats it as a probability — a cost calculation, a threshold set to balance expected losses (lesson 4, Section 7.2), or a risk quoted to the person affected.</li>
    </ul>
    </p>
</details>

**21. The usual complaint about Naive Bayes is that it is wildly over-confident, yet on the interacting data it barely reaches 0.567 when it is right. Resolve the apparent contradiction.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>They are two faces of one broken assumption. When features are <b>correlated</b>, multiplying their probabilities counts the same evidence repeatedly, and the model reports 0.999 with an accuracy nothing like that.</li>
        <li>On the interacting data there is no per-feature evidence at all, so no factor dominates and every posterior sits near the prior — <b>under-confident</b> where the correlated case is over-confident.</li>
        <li>Either way the number is <b>not a probability</b>. "Uncalibrated" does not mean "too high"; it means the map from the reported number to the observed frequency is unknown, and it can err in either direction.</li>
    </ul>
    </p>
</details>

**22. A colleague proposes Naive Bayes for a text classifier: 30,000 word-count features and 4,000 documents. They concede the independence assumption is transparently false — "New" is not independent of "York" given the topic. Should they use it anyway?**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li><b>Yes.</b> This is the canonical case: very high dimensions with little data per feature, where the alternative to a wrong density model is not a better one but <b>no estimate at all</b>.</li>
        <li>Being roughly right in thirty thousand dimensions beats being unable to estimate anything, and the errors from the false assumption largely cancel in the argmax even while they wreck the probabilities.</li>
        <li>It also trains in a single pass, which makes it the <b>baseline to beat</b>: if a tuned model cannot beat Naive Bayes, you have learned that quickly and cheaply.</li>
        <li>Refuse it in two situations only: when the signal is an <b>interaction</b> between features, and when a <b>calibrated probability</b> is required.</li>
    </ul>
    </p>
</details>

## Part 6 — Margins, support vectors and kernels

**23. On separable data every separating line has zero training error. Explain why that makes "minimise the training error" insufficient, and state the criterion a support vector machine uses instead.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>With infinitely many zero-error boundaries the error <b>cannot break the tie</b>. Logistic regression breaks it with the log loss, which is one answer among several rather than the answer.</li>
        <li>The SVM chooses the <b>widest margin</b>: push a slab out from the boundary until it touches the nearest point of each class, and keep the boundary that makes the slab thickest.</li>
        <li>The intuition: a boundary passing close to a training point is <b>one small perturbation away from getting it wrong</b>. Maximising the distance to the closest points picks the boundary that tolerates the most movement in the data before it changes its mind.</li>
        <li>That is a statement about <b>generalisation, not about fit</b>, which is what makes it a genuinely different criterion rather than a tie-break dressed up as one.</li>
    </ul>
    </p>
</details>

**24. In the margin figure, 3 of 80 points determine the boundary. Define a support vector, and give one consequence for the size of the model and one for its fragility.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>A <b>support vector</b> is a training point lying on or inside the margin. Only those points enter the solution; move any of the others and the boundary does not shift at all.</li>
        <li>Consequence for size: the fitted model <i>is</i> those points and their weights, so it is <b>small and fast at prediction</b> — the exact opposite of k-NN, which has to keep everything.</li>
        <li>Consequence for fragility: the model is decided by the points <b>nearest the boundary</b>, which are precisely the ambiguous and the mislabelled ones. A flipped label deep inside a class is harmless; a flipped label at the margin is not.</li>
    </ul>
    </p>
</details>

**25. The hard-margin problem has no solution on the pumps. Describe why, describe the fix, and explain why large C means less regularisation rather than more.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>With 4% of the labels flipped the classes are <b>not separable</b>, so the constraints <code>y_i (w'x_i + b) >= 1</code> cannot all be satisfied at once and the optimisation is infeasible — not merely poor, but with no solution at all.</li>
        <li>The fix is the <b>soft margin</b>: allow violations <code>xi_i >= 0</code>, relax the constraint to <code>y_i (w'x_i + b) >= 1 - xi_i</code>, and add <code>C * sum_i xi_i</code> to the objective.</li>
        <li><b>C is the price of a training error.</b> Large C makes violations expensive, so the model contorts itself to classify everything: narrow margin, low bias, high variance — that is <i>less</i> regularisation, however large the number looks. Small C buys a wider, calmer boundary at the cost of some errors.</li>
        <li>So C runs <b>opposite</b> to lesson 3's regularisation strength; it behaves like <code>1/lambda</code>. It is the same dial as k in Part 2, in a third costume, and the direction is what catches people out.</li>
    </ul>
    </p>
</details>

**26. The linear SVM used 947 of 1,200 points as support vectors (79%) and scored 0.613 ± 0.000; the RBF SVM used 278 (23%) and scored 0.947. Explain why the support-vector fraction is a diagnostic you get for nothing.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>79% means <b>almost every training point sits on or inside the margin</b>. There is no slab that separates anything, so the model is paying slack on most of the dataset.</li>
        <li>It says the same thing as the accuracy, in another language — and it is <b>already computed</b>, so no extra fit and no extra data are needed to read it.</li>
        <li><b>A high support-vector fraction is a free warning</b> that the model is struggling to find room: the wrong kernel, a C set far too small, or classes that genuinely overlap.</li>
        <li>The RBF's 23% is the contrast. A boundary with space around it needs few points to pin it down.</li>
    </ul>
    </p>
</details>

**27. Explain the kernel trick, and state what the support vector machine never has to compute.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>Map the data into a space <code>phi(x)</code> where a linear boundary works, and run the linear method there. Our pumps become separable the moment a third coordinate — <b>distance from the design point</b> — is added: the healthy disc rises a little, the faulty annulus a lot, and a flat plane divides them. <b>The classes were always separable; they needed different coordinates.</b></li>
        <li>The obstacle is that useful spaces are enormous, sometimes infinite-dimensional, so computing <code>phi(x)</code> is out of the question.</li>
        <li>The trick: the SVM's solution depends on the data <b>only through inner products between pairs of points</b>, and a kernel <code>K(x, x') = dot(phi(x), phi(x'))</code> returns that inner product in the new space while computing only with the original coordinates. <b>phi(x) is never computed.</b></li>
        <li>The radial basis function kernel <code>K(x, x') = exp(-gamma * ||x - x'||^2)</code> corresponds to an infinite-dimensional space at a cost of one exponential per pair. Note that we chose the lift in the figure by <b>knowing the answer</b>; the RBF kernel achieves an equivalent effect without being told, which is why it works where nobody could guess the right coordinates.</li>
    </ul>
    </p>
</details>

**28. γ = 50 with C = 1000 gives training 0.995 and cross-validated 0.902 — the highest training score of the three settings and the lowest honest one. Diagnose it, and say what choosing on the training score would have done.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>Textbook <b>overfitting</b>, and here it is visible as well as measurable: the boundary has broken into bubbles around individual points, including the mislabelled ones.</li>
        <li><b>gamma</b> sets how far a single training point's influence reaches. Small gamma: wide reach, smooth boundary. Large gamma: each point influences only its immediate neighbourhood, and the boundary dissolves into islands.</li>
        <li>C = 1000 compounds it, making every violation expensive, so the model has both the means and the motive to enclose each noisy point rather than accept it as an error.</li>
        <li>Choosing on the training score would have selected <b>this</b> model over gamma = 1, C = 1 (training 0.950, cross-validated <b>0.944</b>) — the worst of the three, chosen with complete confidence. Read the two columns together, as lesson 5 taught.</li>
    </ul>
    </p>
</details>

## Part 7 — Choosing a family

**29. On identical data the models in this lesson span 0.613 to 0.947, a gap of 0.334. State the practical conclusion, and contrast it with what tuning buys.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>The three (gamma, C) settings shown for the RBF SVM span 0.902 to 0.944 — about four points. The <b>choice of family</b> spans thirty-three.</li>
        <li>So: <b>choosing the right family matters far more than tuning the wrong one.</b> No amount of tuning rescues a linear model on the pumps, because 0.613 is what the best straight line achieves, not what a badly tuned one achieves.</li>
        <li>The way to tell which family you need is to <b>look at the data first</b>, as lesson 2 insisted. One scatter plot answers the question here, and it costs a minute against the hours a hyperparameter search costs.</li>
    </ul>
    </p>
</details>

**30. A colleague has 900 labelled emails, 25,000 word-count features, and needs a probability of spam to feed a cost calculation that decides whether to quarantine. Which family would you reach for, and what caveat must you attach?**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>For the <b>classification</b>: Naive Bayes. Very many features with little data is the one situation where its assumption being false costs less than the alternative of estimating a 25,000-dimensional density.</li>
        <li>For the <b>probability</b>: not Naive Bayes. Its posteriors are uncalibrated, and word counts are strongly correlated within a document, so here it would fail in the <b>over-confident</b> direction — 0.999 attached to an accuracy nothing like it.</li>
        <li>So use it to <b>rank</b>. The number that gets multiplied by a cost has to come from somewhere else, or be calibrated against held-out data first; the lesson's rule is blunt — <i>need a calibrated probability: not Naive Bayes</i>.</li>
        <li>And whichever model supplies it, the threshold is chosen on validation data and never on the test set (lesson 5).</li>
    </ul>
    </p>
</details>

**31. A second colleague has ten million rows, eight features, a boundary that is plainly not straight, and a service that must answer a thousand queries a second from a small deployed artefact. Choose a family, and say which one you would rule out first.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li><b>Rule out k-NN.</b> Its model is the dataset, so ten million rows would have to ship with the service, and a naive query is O(mn) against all of them. Eight features is comfortable and the boundary shape suits it perfectly — it fails on <b>deployment</b>, not on accuracy, which is the kind of failure a cross-validated score never shows you.</li>
        <li><b>Reach for an RBF SVM.</b> The kernel handles the curved boundary, and the fitted model is only its support vectors — 23% on the pumps, and typically far fewer on cleaner data — so it is small and fast at prediction.</li>
        <li>Watch the support-vector fraction as the free warning it is: if it comes back near 79%, the kernel or C is wrong and the "small model" argument has evaporated.</li>
        <li>Two honest caveats: if k-NN really is wanted, the route is an approximate nearest-neighbour index, which is an engineering commitment rather than a hyperparameter; and the lesson's table compares <i>prediction</i> cost — fitting a kernel SVM on ten million rows is its own problem, which this lesson does not address.</li>
    </ul>
    </p>
</details>

**32. In one sentence, what does this lesson say determines whether a model works?**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li><b>Whether the shape of boundary the family can draw matches the shape the data actually has</b> — and nothing else in this lesson moved the score anywhere near as much.</li>
        <li>The pumps make the point twice over: three unrelated routes to about 0.94 — remembering the neighbourhood, assuming independence given the class, bending the space — and two failures at exactly 0.613, both of them linear.</li>
        <li>The working habit that follows: <b>look at the data, decide what shape the boundary must be, then choose the family.</b> Tuning comes afterwards and buys far less.</li>
    </ul>
    </p>
</details>